In [ ]:
"""
tests/make_synthetic_bronze.py - small synthetic dataset for a pipeline smoke test.

    PIPELINE_ROOT=./_smoke python tests/make_synthetic_bronze.py           # writes data/bronze/*.parquet
    PIPELINE_ROOT=./_smoke python run_pipeline.py --from 3a
    PIPELINE_ROOT=./_smoke python tests/make_synthetic_bronze.py --csv     # writes data/raw/*.csv instead
    PIPELINE_ROOT=./_smoke python run_pipeline.py                          # full run incl. Step 2 ingestion

Change log (see decisions.md D12): --csv mode (tests Step 2), 'fee' transactions (tests fee features).

Schema mirrors the four source files; values follow business_rules.py so the strict mapping
check passes. It also injects a few dirty rows (duplicates, orphans, out-of-window timestamps,
non-integer CSAT) so the Step 3b quarantine / fix logic is exercised. NOT real data.
"""

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().parent))
import duckdb  # noqa: E402

ROOT = Path(os.getenv("PIPELINE_ROOT", "."))
BRONZE = ROOT / "data" / "bronze"
RAW = ROOT / "data" / "raw"
N_CUSTOMERS = int(os.getenv("SMOKE_CUSTOMERS", 50_000))

## 2. Generate customers, transactions, engagements, interactions (+ injected dirty rows)

In [ ]:
def main(as_csv: bool = False) -> None:
    BRONZE.mkdir(parents=True, exist_ok=True)
    con = duckdb.connect()
    con.execute("SELECT setseed(0.42)")
    con.execute(f"""
        CREATE TABLE customers AS
        SELECT 'C' || lpad(i::VARCHAR, 7, '0') AS customer_id,
               DATE '2018-01-01' + CAST(random() * 3100 AS INT) AS signup_date,
               18 + CAST(random() * 60 AS INT) AS age,
               CAST(random() * 120 AS INT) AS tenure_months,
               ['Female','male ','F','M'][1 + CAST(floor(random()*4) AS INT)] AS gender,
               ['north','south','east','west'][1 + CAST(floor(random()*4) AS INT)] AS region,
               'uk' AS country,
               CASE WHEN random() < 0.7 THEN 'consumer' ELSE 'business' END AS customer_segment,
               ['web','store','partner'][1 + CAST(floor(random()*3) AS INT)] AS acquisition_channel,
               ['low','mid','high'][1 + CAST(floor(random()*3) AS INT)] AS income_band,
               ['email','sms','app'][1 + CAST(floor(random()*3) AS INT)] AS preferred_channel,
               ['bronze','silver','gold'][1 + CAST(floor(random()*3) AS INT)] AS loyalty_tier,
               ['Y','N','',NULL][1 + CAST(floor(random()*4) AS INT)] AS marketing_consent,
               'customers.csv' AS _source_file, now()::TIMESTAMP AS _ingested_at
        FROM range(1, {N_CUSTOMERS} + 1) t(i)""")
    con.execute("""
        CREATE TABLE transactions AS
        WITH b AS (SELECT c.customer_id, TIMESTAMP '2025-10-01' + to_seconds(CAST(random() * 31449600 AS BIGINT)) AS ts,
                          random() AS r1, random() AS r2
                   FROM customers c, range(0, 1 + CAST(random() * 30 AS INT)) k(j))
        SELECT 'T' || row_number() OVER () AS transaction_id, customer_id, ts AS transaction_ts,
               strftime(ts, '%Y-%m') AS month,
               CASE WHEN r1 < 0.45 THEN 'purchase' WHEN r1 < 0.75 THEN 'subscription_payment'
                    WHEN r1 < 0.93 THEN 'bill_payment' WHEN r1 < 0.97 THEN 'refund' ELSE 'fee' END AS transaction_type,
               CASE WHEN r2 < 0.90 THEN 'completed' WHEN r2 < 0.97 THEN 'failed' ELSE 'reversed' END AS transaction_status,
               round(5 + random() * 200, 2) AS amount, round(random() * 5, 2) AS discount_amount,
               CASE WHEN random() < 0.9 THEN 'GBP' ELSE 'EUR' END AS currency,
               ['tv','broadband','mobile'][1 + CAST(floor(random()*3) AS INT)] AS product_category,
               ['card','dd'][1 + CAST(floor(random()*2) AS INT)] AS payment_method,
               ['online','store'][1 + CAST(floor(random()*2) AS INT)] AS merchant_channel,
               CASE WHEN random() < 0.05 THEN 'Y' ELSE 'N' END AS is_international,
               'transactions.csv' AS _source_file, now()::TIMESTAMP AS _ingested_at
        FROM b WHERE random() < 0.9""")
    con.execute("""
        CREATE TABLE engagements AS
        WITH b AS (SELECT c.customer_id, TIMESTAMP '2025-10-01' + to_seconds(CAST(random() * 31449600 AS BIGINT)) AS ts
                   FROM customers c, range(0, CAST(random() * 40 AS INT)) k(j))
        SELECT 'E' || row_number() OVER () AS engagement_id, customer_id, ts AS event_ts, strftime(ts, '%Y-%m') AS month,
               ['login','feature_use','offer_redeem','email_click','push_open','sms_click','sms_reply',
                'page_view','offer_view','email_open','session_end','unsubscribe','sms_optout','push_dismiss']
                   [1 + CAST(floor(random()*14) AS INT)] AS event_type,
               ['web','mobile_app','email'][1 + CAST(floor(random()*3) AS INT)] AS engagement_channel,
               ['desktop','ios','android'][1 + CAST(floor(random()*3) AS INT)] AS device_type,
               CAST(random() * 3600 AS INT) AS session_duration_sec, CAST(random() * 20 AS INT) AS pages_viewed,
               CASE WHEN random() < 0.1 THEN 'Y' ELSE 'N' END AS is_converted,
               CASE WHEN random() < 0.3 THEN 'CMP' || CAST(random()*20 AS INT) END AS campaign_id,
               NULL::VARCHAR AS app_version,
               'engagements.csv' AS _source_file, now()::TIMESTAMP AS _ingested_at
        FROM b""")
    con.execute("""
        CREATE TABLE interactions AS
        WITH b AS (SELECT c.customer_id, TIMESTAMP '2025-10-01' + to_seconds(CAST(random() * 31449600 AS BIGINT)) AS ts,
                          random() AS r
                   FROM customers c, range(0, CAST(random() * 6 AS INT)) k(j))
        SELECT 'I' || row_number() OVER () AS interaction_id, customer_id, ts AS interaction_ts,
               strftime(ts, '%Y-%m') AS month,
               ['phone','chat','email'][1 + CAST(floor(random()*3) AS INT)] AS channel,
               ['complaint','billing_dispute','cancellation_request','query','technical_support']
                   [1 + CAST(floor(random()*5) AS INT)] AS interaction_type,
               ['resolved','unresolved','escalated','pending'][1 + CAST(floor(random()*4) AS INT)] AS resolution_status,
               CASE WHEN random() < 0.6 THEN 'Y' ELSE 'N' END AS first_contact_resolution,
               CASE WHEN random() < 0.3 THEN 1 + CAST(floor(random()*5) AS INT) END::DOUBLE AS csat_score,
               CASE WHEN random() < 0.2 THEN CAST(floor(random()*11) AS INT) END::DOUBLE AS nps_score,
               CAST(random() * 1800 AS INT) AS duration_sec,
               CASE WHEN r < 0.3 THEN 'negative' WHEN r < 0.6 THEN 'neutral' ELSE 'positive' END AS sentiment_label,
               round(CASE WHEN r < 0.3 THEN -random() WHEN r < 0.6 THEN 0 ELSE random() END, 3) AS sentiment_score_gt,
               'synthetic text ' || r AS interaction_text, 'A' || CAST(random()*50 AS INT) AS agent_id,
               'interactions.csv' AS _source_file, now()::TIMESTAMP AS _ingested_at
        FROM b""")
    # dirty rows
    con.execute("INSERT INTO transactions SELECT * FROM transactions LIMIT 50")                       # exact duplicates
    con.execute("UPDATE transactions SET customer_id = 'C_ORPHAN' WHERE transaction_id IN ('T1','T2')")
    con.execute("UPDATE engagements SET event_ts = TIMESTAMP '2027-01-01' WHERE engagement_id = 'E5'")
    con.execute("UPDATE interactions SET csat_score = 3.5 WHERE interaction_id IN ('I3','I4')")
    if as_csv:      # raw CSVs in data-dictionary column order (Step 2 checks the header)
        from step2_data_ingestion import SOURCES
        RAW.mkdir(parents=True, exist_ok=True)
        for t, cfg in SOURCES.items():
            cols = ", ".join(cfg["columns"])
            con.execute(f"COPY (SELECT {cols} FROM {t}) TO '{(RAW / cfg['file']).as_posix()}' (HEADER, DELIMITER ',')")
            print(t, con.execute(f"SELECT count(*) FROM {t}").fetchall()[0][0], "->", RAW / cfg["file"])
        return
    for t in ("customers", "transactions", "engagements", "interactions"):
        con.execute(f"COPY {t} TO '{(BRONZE / (t + '.parquet')).as_posix()}' (FORMAT parquet, COMPRESSION zstd)")
        print(t, con.execute(f"SELECT count(*) FROM {t}").fetchall()[0][0])

In [ ]:
if __name__ == "__main__":
    main(as_csv="--csv" in sys.argv)